# TF-IDF + LSI

In [ ]:
# ============================================================
# 03 - TF-IDF + LSI
# ============================================================

import nltk
import pandas as pd
import string
import math
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

# ============================================================
# DOCUMENTS
# ============================================================
documents = [
    "I like mutton biryani.",
    "Chicken curry tastes good with mutton fry.",
    "I enjoy grilled chicken.",
    "Fish fry is tasty.",
    "I am a non vegetarian foodie."
]

# ============================================================
# PREPROCESSING
# ============================================================

stop_words = set(stopwords.words("english"))
processed_docs = []

for doc in documents:
    doc = doc.lower()
    doc = doc.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(doc)
    tokens = [w for w in tokens if w not in stop_words]
    processed_docs.append(tokens)

print("Processed Documents\n")
for i, doc in enumerate(processed_docs):
    print(f"D{i+1}: {' '.join(doc)}")

# ============================================================
# VOCABULARY
# ============================================================

vocab = sorted(set(word for doc in processed_docs for word in doc))

print("\nVocabulary:\n")
print(vocab)

# ============================================================
# TF TABLE
# TF = count(word in document) / total words in document
# ============================================================

tf_table = pd.DataFrame(
    0.0,
    index=vocab,
    columns=[f"D{i+1}" for i in range(len(processed_docs))]
)

for i, doc in enumerate(processed_docs):
    total = len(doc)

    for word in vocab:
        tf = doc.count(word) / total
        tf_table.loc[word, f"D{i+1}"] = round(tf, 3)

print("\n====================")
print("TERM FREQUENCY (TF)")
print("====================\n")
print(tf_table)

# ============================================================
# IDF TABLE
# IDF = log(N / DF)
# ============================================================

N = len(processed_docs)

idf_values = {}

for word in vocab:
    df = sum(word in doc for doc in processed_docs)
    idf = math.log(N / df)
    idf_values[word] = round(idf, 3)

idf_table = pd.DataFrame.from_dict(
    idf_values,
    orient="index",
    columns=["IDF"]
)

print("\n====================")
print("IDF TABLE")
print("====================\n")
print(idf_table)

# ============================================================
# TF-IDF TABLE
# ============================================================

tfidf_table = tf_table.copy()

for word in vocab:
    tfidf_table.loc[word] = (
        tf_table.loc[word] * idf_values[word]
    ).round(3)

print("\n====================")
print("TF-IDF = TF × IDF")
print("====================\n")
print(tfidf_table)

# ============================================================
# TF-IDF MATRIX
# sklearn expects:
# rows = documents
# columns = terms
# ============================================================

tfidf_matrix = tfidf_table.T.values

print("\n====================")
print("TF-IDF MATRIX")
print("====================\n")
print(tfidf_matrix)

# ============================================================
# LSI USING TRUNCATED SVD
# ============================================================

# Number of latent dimensions.
requested_components = 2

# TruncatedSVD cannot use as many components as the number
# of features in some small datasets.
max_components = min(tfidf_matrix.shape[0], tfidf_matrix.shape[1] - 1)

if max_components < 1:
    print("\nNot enough data/features for TruncatedSVD.")
else:
    n_components = min(requested_components, max_components)

    svd = TruncatedSVD(
        n_components=n_components,
        random_state=42
    )

    document_lsi = svd.fit_transform(tfidf_matrix)

    print("\n====================")
    print("LSI COMPONENTS")
    print("====================\n")
    print(svd.components_)

    print("\n====================")
    print("DOCUMENT REPRESENTATION IN LSI SPACE")
    print("====================\n")

    lsi_table = pd.DataFrame(
        document_lsi,
        index=[f"D{i+1}" for i in range(len(documents))],
        columns=[f"Topic{i+1}" for i in range(n_components)]
    )
    print(lsi_table)

    print("\nExplained variance ratio:")
    print(svd.explained_variance_ratio_)

    # ========================================================
    # QUERY IN THE SAME LSI SPACE
    # ========================================================

    query = "mutton chicken fry"

    query_tokens = word_tokenize(query.lower())
    query_tokens = [
        w for w in query_tokens
        if w not in stop_words and w in vocab
    ]

    query_tf = []
    for word in vocab:
        query_tf.append(
            query_tokens.count(word) / len(query_tokens)
            if len(query_tokens) > 0 else 0
        )

    query_tfidf = [
        query_tf[i] * idf_values[word]
        for i, word in enumerate(vocab)
    ]

    query_lsi = svd.transform([query_tfidf])

    print("\nQuery:", query)
    print("Query LSI representation:")
    print(query_lsi)

    similarities = cosine_similarity(query_lsi, document_lsi)[0]

    print("\nCosine similarity with documents:")
    for i, score in enumerate(similarities):
        print(f"D{i+1}: {score:.3f}")

# ============================================================
# PIPELINE TO REMEMBER
# ============================================================
# Documents
#     ↓
# Lowercase
#     ↓
# Remove punctuation
#     ↓
# Tokenize
#     ↓
# Remove stopwords
#     ↓
# Vocabulary
#     ↓
# TF
#     ↓
# IDF
#     ↓
# TF-IDF
#     ↓
# LSI / TruncatedSVD
#
# IMPORTANT:
# tfidf_table is [terms x documents]
# sklearn SVD input here is [documents x terms],
# therefore we use tfidf_table.T

# ============================================================
# ============================================================
# documents = [...]
#
# requested_components = 2
#
# query = "..."


Processed Documents

D1: like mutton biryani
D2: chicken curry tastes good mutton fry
D3: enjoy grilled chicken
D4: fish fry tasty
D5: non vegetarian foodie

Vocabulary:

['biryani', 'chicken', 'curry', 'enjoy', 'fish', 'foodie', 'fry', 'good', 'grilled', 'like', 'mutton', 'non', 'tastes', 'tasty', 'vegetarian']

TERM FREQUENCY (TF)

                          D1            D2            D3            D4            D5
       biryani         0.333         0.000         0.000         0.000         0.000
       chicken         0.000         0.167         0.333         0.000         0.000
         curry         0.000         0.167         0.000         0.000         0.000
         enjoy         0.000         0.000         0.333         0.000         0.000
          fish         0.000         0.000         0.000         0.333         0.000
        foodie         0.000         0.000         0.000         0.000         0.333
           fry         0.000         0.167         0.000         0.33